# E-ZERO: the runs that still need a GPU

This notebook runs the benchmark stages that are not finished yet. It runs the **same
commands as the Kaggle kernels, from the same commit**, so a run here is interchangeable
with a run there: results are keyed by model, dataset, horizon and protocol, and merged
by the same script.

| Stage | What it measures | Roughly, on one T4 |
|---|---|---|
| `tsfmfix` | Moirai-S re-run after the median fix; Lag-Llama on the five datasets it can finish | 1 h + 6 h |
| `tsfmval` | the foundation models under the protocols their own papers used | ~4 h |
| `fullval` | the full-shot baselines with their published per-dataset settings | ~6 h |
| `fullshot2` | TSMixer, TimeMixer, TimesNet trained per dataset and horizon | ~5 h |
| `fullshot3` | FEDformer, Autoformer, the same way | ~6 h |

**Nothing is lost if the session is cut off.** Everything produced is written to a folder
in your Drive as it goes, and the next run copies it back and skips the work that is
already done. Run the same cell again after a disconnect and it carries on.

Before you start: **Runtime -> Change runtime type -> T4 GPU**.


## 1. The machine


In [ ]:
!nvidia-smi
import torch, sys
print(sys.version.split()[0], 'torch', torch.__version__, 'cuda', torch.cuda.is_available())


## 2. Where the results are kept

Mount your Drive. Everything goes into `MyDrive/ezero-runs/`, and that folder is what you
send back - it is the only thing that has to survive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

OUT = '/content/drive/MyDrive/ezero-runs'   # change if you like
import os; os.makedirs(OUT, exist_ok=True)
print(OUT)


## 3. The code

Cloned at a branch, then installed. Re-running this cell is safe.


In [ ]:
BRANCH = 'ezero-v2-paper1-structure-kaggle'
REPO = 'https://github.com/r2barati/ezero.git'
SRC  = '/content/ezero'

import os, subprocess
if not os.path.isdir(SRC):
    !git clone --quiet $REPO $SRC
!git -C $SRC fetch --quiet origin $BRANCH && git -C $SRC checkout --quiet $BRANCH && git -C $SRC pull --quiet
!pip install --quiet -e $SRC
!git -C $SRC log --oneline -1


## 4. The evaluation data

ETTh1, ETTh2, ETTm1 and ETTm2 are downloaded straight from the dataset's own GitHub.

**Weather, Electricity and Traffic are not on a public URL.** Two ways to get them:

* **Kaggle**, which the next cell will use if it finds credentials: go to
  kaggle.com -> your profile -> Settings -> API -> *Create New Token*, which downloads
  `kaggle.json`, and upload it when the cell asks.
* **By hand**: put `weather.csv`, `electricity.csv` and `traffic.csv` into
  `MyDrive/ezero-runs/raw/` and the cell will pick them up from there.

`fullshot2`, `fullshot3`, `fullval` and `tsfmval` need all seven. `tsfmfix` needs Weather
for Lag-Llama and all seven for Moirai-S.


In [ ]:
import os, shutil, glob

RAW = os.path.join(SRC, 'data', 'raw'); os.makedirs(RAW, exist_ok=True)
DRIVE_RAW = os.path.join(OUT, 'raw'); os.makedirs(DRIVE_RAW, exist_ok=True)
for p in glob.glob(os.path.join(DRIVE_RAW, '*.csv')):      # anything you put in Drive
    shutil.copy2(p, RAW)

if not os.path.exists('/root/.kaggle/kaggle.json'):
    print('No Kaggle token found. Upload kaggle.json now, or press Cancel to skip.')
    try:
        from google.colab import files
        up = files.upload()
        if 'kaggle.json' in up:
            os.makedirs('/root/.kaggle', exist_ok=True)
            open('/root/.kaggle/kaggle.json', 'wb').write(up['kaggle.json'])
            os.chmod('/root/.kaggle/kaggle.json', 0o600)
    except Exception as e:
        print('skipped:', e)

os.chdir(SRC)
from ezero.data.ltsf import LSF_DATASETS, fetch, find_file
for name in LSF_DATASETS:
    try:
        print(f'{name:12s} {fetch(name)}')
    except Exception as e:
        print(f'{name:12s} MISSING - {type(e).__name__}: {str(e)[:110]}')

for p in glob.glob(os.path.join(RAW, '*.csv')):            # keep a copy in Drive
    if not os.path.exists(os.path.join(DRIVE_RAW, os.path.basename(p))):
        shutil.copy2(p, DRIVE_RAW)


## 5. Run a stage

Set `STAGE` and run. `HOURS` is when it stops starting new work - set it a little under
how long you expect the session to last, so the last command is not killed mid-write.

For `tsfmfix`, run it twice, once per model, so a disconnect during Lag-Llama does not
cost you the Moirai-S result:

```
STAGE = 'tsfmfix';  TSFM_MODELS = 'Moirai-S'
STAGE = 'tsfmfix';  TSFM_MODELS = 'Lag-Llama'
```


In [ ]:
STAGE       = 'tsfmfix'     # tsfmfix | tsfmval | fullval | fullshot2 | fullshot3
TSFM_MODELS = 'Moirai-S'   # tsfm stages only: a comma list, or 'ALL'
HOURS       = 3.5          # stop starting new commands after this long

!cd $SRC && python colab/run_stage.py --stage $STAGE --tsfm-models $TSFM_MODELS \
    --hours $HOURS --out $OUT


## 6. What came out

One CSV per model under `results/cache/`. Re-running a stage skips every model and dataset
already listed here.


In [ ]:
import os, pandas as pd

cache = os.path.join(OUT, 'working', 'outputs', 'results', 'cache')
rows = []
for f in sorted(os.listdir(cache)) if os.path.isdir(cache) else []:
    if not f.endswith('.csv'):
        continue
    d = pd.read_csv(os.path.join(cache, f))
    rows.append({'file': f, 'rows': len(d),
                 'datasets': ', '.join(sorted(d['dataset'].unique())) if 'dataset' in d else '',
                 'protocols': ', '.join(sorted(d['protocol'].astype(str).unique())) if 'protocol' in d else ''})
pd.set_option('display.width', 200); pd.set_option('display.max_colwidth', 80)
print(pd.DataFrame(rows).to_string(index=False) if rows else 'nothing yet')


## 7. Sending it back

Share the `ezero-runs` folder, or zip it and attach it. The useful part is
`working/outputs/` - the caches, the training logs and the per-stage `*.log` files.

```python
!cd '/content/drive/MyDrive' && zip -qr ezero-runs.zip ezero-runs/working && ls -lh ezero-runs.zip
```
